<a href="https://colab.research.google.com/github/shivaprajapati34390-netizen/ML-project/blob/main/Predictive_keyboard_model_using_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


step 1 Prepareing the datasets

In [1]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
# load the datasherlock-holm
with open("/content/sample_data/sherlock-holm.es_stories_plain-text_advs","r",encoding="utf-8") as f:
    text=f.read().lower()

nltk.download('punkt_tab')
tokens=word_tokenize(text)
print("Total Tokens : ",len(tokens))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total Tokens :  125772



step 2 creating the vocabulary

In [3]:
from collections import Counter

In [4]:
word_counts=Counter(tokens)
vocab=sorted(word_counts,key=word_counts.get,reverse=True)


In [5]:
word2idx={word: idx for idx,word in enumerate(vocab)}
idx2word={idx: word for word,idx in enumerate(vocab)}
vocab_size=(len(vocab))


step 3 building input output sequences

In [6]:
from typing import Sequence
Sequence_length=4 # e.g., "I am going to [predict this]"



In [7]:
data=[]
for i in range (len(tokens)-Sequence_length):
  input_seq=tokens[i:i+Sequence_length-1]
  target_word=tokens[i+ Sequence_length-1]
  data.append((input_seq,target_word))



In [8]:
import torch
# convert word to indices
def encode(seq): return [word2idx[word] for word in seq]

encoded_data=[(torch.tensor(encode(inp)),torch.tensor(word2idx[target])) for inp,target in data]


step 4 Designing the model architecture

In [9]:
import torch.nn as nn


In [10]:
class PredictKeyboard(nn.Module):
  def __init__(self,vocab_size,embed_dim=64,hidden_dim=128):
    super(PredictKeyboard,self).__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim)
    self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True)
    self.fc=nn.Linear(hidden_dim,vocab_size)

  def forward(self,x):
    x=self.embedding(x)
    output, _ = self.lstm(x) # Correctly unpack output and discard hidden/cell states
    output=self.fc(output[:,-1,:]) #last lstm output
    return output

step-5 traning the model

In [11]:
import torch
import torch.optim as optim
import random

In [12]:
model=PredictKeyboard(vocab_size)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.005)


In [13]:
epochs=20
for epoch in range(epochs):
  total_loss=0
  random.shuffle(encoded_data)
  for input_seq,target in encoded_data[:10000]: #limit data for speed
   input_seq=input_seq.unsqueeze(0)
   output=model(input_seq)
   loss=criterion(output,target.unsqueeze(0))
   optimizer.zero_grad()
   loss.backward()
   optimizer.step()
   total_loss+=loss.item()

KeyboardInterrupt: 

In [ ]:
print("f epoch{epoch+1}.loss:{total_loss:.4f}")